In [0]:
from pyspark.sql.functions import *

In [0]:
bronze_df = spark.read.table("bronze_stock_data")

bronze_df.show(5)

+-------------------+------------------+------------------+------------------+------------------+--------+---------+------------+--------------------+--------------+
|               Date|              Open|              High|               Low|             Close|  Volume|Dividends|Stock_Splits| ingestion_timestamp|ingestion_date|
+-------------------+------------------+------------------+------------------+------------------+--------+---------+------------+--------------------+--------------+
|2026-02-02 05:00:00| 259.7869174093499| 270.2371306501278| 258.9676766447768| 269.7575988769531|73913400|      0.0|         0.0|2026-03-17 12:09:...|    2026-03-17|
|2026-02-03 05:00:00| 268.9483513556569| 271.6258386480508| 267.3598109321867|269.22808837890625|64394700|      0.0|         0.0|2026-03-17 12:09:...|    2026-03-17|
|2026-02-04 05:00:00|   272.03545112075|278.68922850453845|   272.03545112075|276.23150634765625|90545700|      0.0|         0.0|2026-03-17 12:09:...|    2026-03-17|
|202

In [0]:
bronze_df.printSchema()

root
 |-- Date: timestamp (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: long (nullable = true)
 |-- Dividends: double (nullable = true)
 |-- Stock_Splits: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_date: date (nullable = true)



In [0]:
silver_df = bronze_df.dropDuplicates(["Date"])

In [0]:
valid_df = silver_df.filter(
    (col("Close") > 0) &
    (col("Volume") > 0) &
    (col("Date").isNotNull())
)

In [0]:
invalid_df = silver_df.filter(
    (col("Close") <= 0) |
    (col("Volume") <= 0) |
    (col("Date").isNull())
)

In [0]:
invalid_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_invalid_records")

In [0]:
silver_df = valid_df

In [0]:
silver_df = silver_df.withColumn("processed_timestamp", current_timestamp())

In [0]:
silver_df = silver_df.select(
    "Date",
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "ingestion_timestamp",
    "ingestion_date",
    "processed_timestamp"
)

In [0]:
silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_stock_data")

In [0]:
%sql
SELECT * FROM silver_stock_data LIMIT 5;

Date,Open,High,Low,Close,Volume,ingestion_timestamp,ingestion_date,processed_timestamp,daily_return,ma_7,ma_30,avg_vol_30,volatility_30
2026-02-17T05:00:00.000Z,258.04998779296875,266.2900085449219,255.5399932861328,263.8800048828125,58469100,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,null,null,null,null,null
2026-02-18T05:00:00.000Z,263.6000061035156,266.82000732421875,262.45001220703125,264.3500061035156,34203300,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,null,null,null,null,null
2026-03-09T04:00:00.000Z,255.69000244140625,261.1499938964844,253.67999267578125,259.8800048828125,38218500,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,null,null,null,null,null
2026-03-10T04:00:00.000Z,257.6499938964844,262.4800109863281,256.95001220703125,260.8299865722656,30590800,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,null,null,null,null,null
2026-02-10T05:00:00.000Z,274.8900146484375,275.3699951171875,272.94000244140625,273.67999267578125,34376900,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,null,null,null,null,null


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

silver_df = spark.read.table("silver_stock_data")

silver_df.show(5)

+-------------------+------------------+------------------+------------------+------------------+--------+--------------------+--------------+--------------------+------------+----+-----+----------+-------------+
|               Date|              Open|              High|               Low|             Close|  Volume| ingestion_timestamp|ingestion_date| processed_timestamp|daily_return|ma_7|ma_30|avg_vol_30|volatility_30|
+-------------------+------------------+------------------+------------------+------------------+--------+--------------------+--------------+--------------------+------------+----+-----+----------+-------------+
|2026-02-17 05:00:00|258.04998779296875| 266.2900085449219| 255.5399932861328| 263.8800048828125|58469100|2026-03-17 12:09:...|    2026-03-17|2026-03-17 12:11:...|        NULL|NULL| NULL|      NULL|         NULL|
|2026-02-18 05:00:00| 263.6000061035156|266.82000732421875|262.45001220703125| 264.3500061035156|34203300|2026-03-17 12:09:...|    2026-03-17|2026-0

In [0]:
window_spec = Window.orderBy("Date")

In [0]:
silver_df = silver_df.withColumn(
    "prev_close",
    lag("Close").over(window_spec)
)

silver_df = silver_df.withColumn(
    "daily_return",
    (col("Close") - col("prev_close")) / col("prev_close")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
window_7 = Window.orderBy("Date").rowsBetween(-6, 0)

silver_df = silver_df.withColumn(
    "ma_7",
    avg("Close").over(window_7)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
window_30 = Window.orderBy("Date").rowsBetween(-29, 0)

silver_df = silver_df.withColumn(
    "ma_30",
    avg("Close").over(window_30)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
silver_df = silver_df.withColumn(
    "avg_vol_30",
    avg("Volume").over(window_30)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
silver_df = silver_df.withColumn(
    "volatility_30",
    stddev("Close").over(window_30)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
silver_df = silver_df.drop("prev_close")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
silver_df.select(
    "Date", "Close", "daily_return", "ma_7", "ma_30", "volatility_30"
).show(10)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------+------------------+--------------------+------------------+------------------+-------------------+
|               Date|             Close|        daily_return|              ma_7|             ma_30|      volatility_30|
+-------------------+------------------+--------------------+------------------+------------------+-------------------+
|2026-02-02 05:00:00| 269.7575988769531|                NULL| 269.7575988769531| 269.7575988769531|               NULL|
|2026-02-03 05:00:00|269.22808837890625|-0.00196291225993...| 269.4928436279297| 269.4928436279297|0.37442046387841144|
|2026-02-04 05:00:00|276.23150634765625| 0.02601295433518634| 271.7390645345052| 271.7390645345052| 3.8995666971211462|
|2026-02-05 05:00:00| 275.6520690917969|-0.00209765085641...| 272.7173156738281| 272.7173156738281| 3.7370641038847494|
|2026-02-06 05:00:00| 277.8599853515625| 0.00800979389358529|  273.745849609375|  273.745849609375|  3.970345875394946|
|2026-02-09 05:00:00| 274.6199951171875|

In [0]:
silver_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_stock_data")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_stock_data")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
%sql
SELECT * FROM silver_stock_data LIMIT 5;

Date,Open,High,Low,Close,Volume,ingestion_timestamp,ingestion_date,processed_timestamp,daily_return,ma_7,ma_30,avg_vol_30,volatility_30
2026-02-02T05:00:00.000Z,259.7869174093499,270.2371306501278,258.9676766447768,269.7575988769531,73913400,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,null,269.7575988769531,269.7575988769531,7.39134E7,null
2026-02-03T05:00:00.000Z,268.9483513556569,271.6258386480508,267.3598109321867,269.22808837890625,64394700,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,-0.001962912259937505,269.4928436279297,269.4928436279297,6.915405E7,0.37442046387841144
2026-02-04T05:00:00.000Z,272.03545112075,278.68922850453845,272.03545112075,276.23150634765625,90545700,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,0.02601295433518634,271.7390645345052,271.7390645345052,7.62846E7,3.8995666971211462
2026-02-05T05:00:00.000Z,277.86999494352693,279.2387093202658,272.97458180817284,275.6520690917969,52977400,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,-0.0020976508564164785,272.7173156738281,272.7173156738281,7.04578E7,3.7370641038847494
2026-02-06T05:00:00.000Z,276.8609202349588,280.6473855638201,276.67109542368036,277.8599853515625,50453400,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,0.00800979389358529,273.745849609375,273.745849609375,6.645692E7,3.970345875394946
